In [ ]:
# Implementation: Orchestrator + Agents  (self-contained)
import re
from typing import Dict
from datetime import datetime, timedelta

# --- Domain models  ---
class BookingSystem:
    def __init__(self):
        self.bookings: Dict[str, Dict[str, str]] = {}

    def check_availability(self, date: str, time: str) -> bool:
        if date not in self.bookings:
            return True
        return time not in self.bookings[date]

    def add_booking(self, date: str, time: str, customer: str) -> bool:
        self.bookings.setdefault(date, {})
        self.bookings[date][time] = customer
        return True

    def get_bookings(self, date: str) -> Dict[str, str]:
        return self.bookings.get(date, {})


booking_system = BookingSystem()


class Inventory:
    def __init__(self):
        self.stock: Dict[str, int] = {
            "skateboard": 20,
            "helmet": 30,
            "wheels": 50,
        }

    def check_stock(self, item: str) -> int:
        return self.stock.get(item, 0)

    def sell_item(self, item: str, quantity: int) -> bool:
        if self.stock.get(item, 0) >= quantity:
            self.stock[item] -= quantity
            return True
        return False


inventory = Inventory()


def get_inventory_level(item: str) -> str:
    level = inventory.check_stock(item)
    return f"{item}:{level}"


def sell_inventory_item(item: str, quantity: int) -> str:
    if inventory.sell_item(item, quantity):
        return f"sold:{quantity}:{item}"
    return f"failed:{item}"


# --- Agents ---
class CustomerSupportAgent:
    def diagnose_issue(self, request: str) -> str:
        req = request.lower()
        if any(k in req for k in ["board", "skateboard", "helmet", "wheels", "buy", "stock"]):
            return "Shop"
        if any(k in req for k in ["rent", "session", "booking", "book", "park"]):
            return "Park"
        if any(k in req for k in ["broken", "damaged", "repair"]):
            return "Repair"
        return "Unknown"

    def initial_response(self, diagnosis: str) -> str:
        mapping = {
            "Shop": "Sure — what item and quantity are you interested in?",
            "Park": "Great — what date (YYYY-MM-DD) and time (HH:MM) would you like?",
            "Repair": "Please describe the damage; we can assess repair or replacement.",
            "Unknown": "Could you clarify your request?",
        }
        return mapping.get(diagnosis, "Could you clarify your request?")


class InventoryAgent:
    def get_level(self, item: str) -> str:
        return get_inventory_level(item)

    def sell(self, item: str, quantity: int) -> str:
        return sell_inventory_item(item, quantity)


class ParkManagementAgent:
    def check(self, date: str, time: str) -> str:
        return "available" if booking_system.check_availability(date, time) else "unavailable"

    def add(self, date: str, time: str, customer: str) -> str:
        booking_system.add_booking(date, time, customer)
        return f"Booking confirmed for {customer} on {date} at {time}."

    def list_bookings(self, date: str) -> Dict[str, str]:
        return booking_system.get_bookings(date)


# --- Orchestrator ---
class Orchestrator:
    def __init__(self):
        self.support = CustomerSupportAgent()
        self.inventory = InventoryAgent()
        self.park = ParkManagementAgent()

    def _suggest_nearby_slot(self, date: str, time: str):
        dt = datetime.strptime(f"{date} {time}", "%Y-%m-%d %H:%M")
        for delta_hours in (1, 2, 3, 4):
            candidate = dt + timedelta(hours=delta_hours)
            cand_date = candidate.strftime("%Y-%m-%d")
            cand_time = candidate.strftime("%H:%M")
            if booking_system.check_availability(cand_date, cand_time):
                return cand_date, cand_time
        next_day = dt + timedelta(days=1)
        return next_day.strftime("%Y-%m-%d"), "09:00"

    def handle_request(self, user_request: str, customer_name: str = "Guest") -> str:
        diagnosis = self.support.diagnose_issue(user_request)
        preface = self.support.initial_response(diagnosis)

        # parsing
        date_match = re.search(r"\b\d{4}-\d{2}-\d{2}\b", user_request)
        time_match = re.search(r"\b\d{2}:\d{2}\b", user_request)
        buy_match = re.search(r"\bbuy\s+(\d+)\s+(\w+)\b", user_request.lower())

        # Park flow
        if diagnosis == "Park":
            if date_match and time_match:
                date = date_match.group(0)
                time = time_match.group(0)
                avail = self.park.check(date, time)
                if avail == "available":
                    confirmation = self.park.add(date, time, customer_name)
                    return f"{preface} {confirmation}"
                else:
                    sug_date, sug_time = self._suggest_nearby_slot(date, time)
                    return (
                        f"{preface} Unfortunately that slot is taken. "
                        f"We can offer {sug_date} at {sug_time} instead — would that work?"
                    )
            return f"{preface}"

        # Shop flow
        if diagnosis == "Shop":
            if buy_match:
                qty = int(buy_match.group(1))
                item = buy_match.group(2)
                result = self.inventory.sell(item, qty)
                if result.startswith("sold"):
                    remaining = inventory.check_stock(item)
                    low_note = ""
                    if remaining < 5:
                        low_note = f" Note: stock for {item} is low ({remaining}) — reorder advised."
                    return f"{preface} {qty} {item} purchased. {result}.{low_note}"
                else:
                    level = self.inventory.get_level(item)
                    return f"{preface} Sorry — we couldn't complete the sale. Current stock: {level}"
            item_search = re.search(r"(skateboard|helmet|wheels)", user_request.lower())
            item = item_search.group(1) if item_search else "skateboard"
            level = self.inventory.get_level(item)
            return f"{preface} Current stock — {level}"

        if diagnosis == "Repair":
            return f"{preface} We'll connect you with our repair specialist in Nairobi."

        return f"{preface}"


# Orchestrating Agent Activities — Skate Park & Shop

This notebook contains a self-contained implementation of the orchestration exercise: CustomerSupportAgent, InventoryAgent, ParkManagementAgent, and an Orchestrator that routes requests and composes responses. The demo below shows sample interactions.

In [ ]:
# Demo cell: sample interactions
orch = Orchestrator()

req1 = "I want to book a skate session for 2025-09-12 at 10:00."
print("Request 1:", req1)
print("Response 1:", orch.handle_request(req1, customer_name="Aisha"))

req2 = "Do you have skateboards? Can I buy 2 skateboards?"
print("\nRequest 2:", req2)
print("Response 2:", orch.handle_request(req2, customer_name="Brian"))

req3 = "My helmet is broken!"
print("\nRequest 3:", req3)
print("Response 3:", orch.handle_request(req3, customer_name="Cynthia"))
